# 03 — Convergent and discriminant audit

This notebook tests the prespecified relationships between reviewed indicators
and external measures.

Convergence is interpreted according to the declared relationship class.
Equality is expected only for true mathematical equivalents. Perceptual
comparators can support perceptual relevance, not physical source attribution.


In [ ]:
from pathlib import Path
import pandas as pd

from paper1_qc.external_validation import (
    convergent_validity_audit,
    discover_project_root,
    load_audit_config,
)

PROJECT_ROOT = discover_project_root()
CONFIG = load_audit_config(PROJECT_ROOT)
paths = convergent_validity_audit(PROJECT_ROOT, CONFIG)
paths


In [ ]:
results = pd.read_csv(paths["results"])
display(
    results[
        [
            "family_code",
            "feature",
            "comparator",
            "comparator_column",
            "relationship_class",
            "expected_direction",
            "n",
            "spearman_rho",
            "ci95_low",
            "ci95_high",
            "bootstrap_probability_expected_direction",
            "p_fdr",
            "direction_matches",
            "availability_status",
        ]
    ].sort_values(["family_code", "feature", "comparator"])
)


## Interpretation rules

- Strong agreement with a related primitive supports convergent validity but
  does not establish equivalence.
- Weak agreement with a perceptual model may be expected for localized or
  conditional waveform observables.
- A direction reversal requires case review before any extractor change.
- Agreement that is driven only by diagnosis, speaker, or recording duration
  is not sufficient. Repeat relevant analyses within cohort and within
  participant where support permits.
- Review the largest disagreements by listening and waveform inspection.


In [ ]:
merged = pd.read_parquet(paths["merged"])

# Export the largest standardized disagreements for mappings with sufficient data.
out_dir = PROJECT_ROOT / "outputs" / "06_external_validation" / "03_convergence"
case_rows = []
for row in results.itertuples(index=False):
    if row.availability_status != "ok":
        continue
    if row.feature not in merged or row.comparator_column not in merged:
        continue
    local = merged[["logical_recording_id", row.feature, row.comparator_column]].copy()
    local[row.feature] = pd.to_numeric(local[row.feature], errors="coerce")
    local[row.comparator_column] = pd.to_numeric(local[row.comparator_column], errors="coerce")
    local = local.dropna()
    if len(local) < 10:
        continue
    for column in [row.feature, row.comparator_column]:
        median = local[column].median()
        mad = (local[column] - median).abs().median()
        scale = 1.4826 * mad if mad > 0 else local[column].std(ddof=0)
        local[f"{column}_z"] = (local[column] - median) / scale if scale > 0 else 0.0
    expected = 1 if row.expected_direction == "positive" else -1
    local["disagreement_score"] = (
        local[f"{row.feature}_z"] - expected * local[f"{row.comparator_column}_z"]
    ).abs()
    selected = local.nlargest(10, "disagreement_score").copy()
    selected["feature"] = row.feature
    selected["comparator"] = row.comparator
    selected["comparator_column"] = row.comparator_column
    case_rows.append(selected)

cases = pd.concat(case_rows, ignore_index=True) if case_rows else pd.DataFrame()
cases.to_csv(out_dir / "largest_disagreement_cases.csv", index=False)
display(cases.head(50))
